# 03 — Throughput Analysis

This notebook addresses **RQ2**: *Which service sustains higher request throughput
under identical load?*

We compare requests-per-second for both services, examine the correlation between
Kafka consumer lag and P95 latency, and test whether observed RPS differences are
statistically significant.

## Imports and setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

sys.path.insert(0, str(Path('..') / 'scripts'))
from utils import SERVICES, SERVICE_LABELS, SERVICE_COLORS, load_csv, describe_series, set_plot_style
from statistical_tests import _compute_test_result, run_throughput_tests
from generate_charts import fig03_throughput_timeseries

set_plot_style()
%matplotlib inline

## Load throughput and consumer lag data

Throughput is measured as the rate of HTTP requests completed per second,
recorded by k6 and pushed to Prometheus via remote-write.
Consumer lag records how many messages are waiting in Kafka partitions,
indicating back-pressure from the downstream consumer.

In [ ]:
rps_dn  = load_csv('throughput_dotnet.csv')['rps']
rps_go  = load_csv('throughput_go.csv')['rps']

try:
    lag_dn = load_csv('consumer_lag_dotnet.csv')
    lag_go = load_csv('consumer_lag_go.csv')
    has_lag = True
except FileNotFoundError:
    has_lag = False
    print('consumer_lag CSVs not found — lag correlation will be skipped')

print(f'.NET RPS samples: {len(rps_dn)}')
print(f'Go   RPS samples: {len(rps_go)}')

## Descriptive statistics: mean/max/min RPS

We summarise steady-state throughput for each service.  The mean RPS reflects
typical throughput; the max and min bound the range observed during the benchmark.

In [ ]:
for label, s in [('.NET', rps_dn), ('Go', rps_go)]:
    d = describe_series(s)
    print(f'{label}: mean={d["mean"]:.1f} rps  max={d["max"]:.1f}  min={d["min"]:.1f}  std={d["std"]:.2f}')

## Mann-Whitney U + effect size on RPS

Higher RPS is better, so we pass `lower_is_better=False`.

In [ ]:
r = _compute_test_result(
    rps_dn.dropna().to_numpy(), rps_go.dropna().to_numpy(),
    'Throughput', 'rps', n_bootstrap=10_000, lower_is_better=False,
)
print(f'p-value     : {r.p_value:.6f}  ({"significant" if r.significant else "not significant"} at α=0.05)')
print(f'.NET mean   : {r.dotnet_mean:.2f} rps')
print(f'Go mean     : {r.go_mean:.2f} rps')
print(f'Difference  : {r.mean_diff:+.2f} rps  (.NET − Go)')
print(f'Effect size : {r.effect_size:.4f}  ({r.effect_label})')
print(f'Winner      : {r.winner.upper()}')

## Bootstrap CI on mean RPS difference

In [ ]:
print(f'95% CI on mean RPS diff (.NET − Go): [{r.ci_low:+.2f}, {r.ci_high:+.2f}] rps')
ci_excludes_zero = not (r.ci_low <= 0 <= r.ci_high)
print(f'CI excludes zero: {ci_excludes_zero}  → consistent directional difference: {ci_excludes_zero}')

## Figure 3: throughput + consumer lag time series

In [ ]:
fig03_throughput_timeseries()

## Correlation: consumer lag vs P95 latency

We test whether Kafka consumer lag is a leading indicator of latency degradation.
A strong positive Pearson r suggests that as messages queue up, the HTTP response
time increases — consistent with back-pressure propagation through the pipeline.

In [ ]:
if has_lag:
    for svc in SERVICES:
        try:
            lag_df = load_csv(f'consumer_lag_{svc}.csv')
            lat_df = load_csv(f'latency_p95_{svc}.csv')
            lag_s = lag_df.groupby(lag_df.index)['lag'].sum() if 'lag' in lag_df.columns else lag_df.iloc[:, 0]
            combined = pd.concat([lag_s.rename('lag'), lat_df['value_ms'].rename('p95')], axis=1).dropna()
            r_val, p_val = stats.pearsonr(combined['lag'], combined['p95'])
            print(f'{SERVICE_LABELS[svc]}: Pearson r={r_val:.4f}  p={p_val:.4f}')
        except (FileNotFoundError, KeyError) as e:
            print(f'  Skipping {svc}: {e}')
else:
    print('Lag data not available.')

## Interpretation

**Fill in after running with real data.**

Template:

> The **Go service achieved X% higher mean RPS** (Mann-Whitney U, p=XXXX,
> Cliff's delta=X.XX, Y effect).  The 95% CI [+A, +B] excludes zero, confirming
> a consistent throughput advantage.
>
> Consumer lag and P95 latency were **[weakly/moderately/strongly] correlated**
> for both services (r=X.XX, p=XXXX), suggesting that Kafka back-pressure
> [does/does not] propagate into HTTP response latency at the tested load levels.